Scraping the data set from PDF

In [1]:
from pathlib import Path
import time
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, RapidOcrOptions, TableStructureOptions, smolvlm_picture_description
from docling.document_converter import DocumentConverter, WordFormatOption, PdfFormatOption
from dotenv import load_dotenv

table_structure_option= TableStructureOptions(
    do_cell_matching = True
)
pipeline_option = PdfPipelineOptions (
    generate_page_images=True,
    images_scale=1.00,
    do_ocr=True,
    do_picture_description=True,
    ocr_options=RapidOcrOptions(),
    do_table_structure=True,
    table_structure_options = table_structure_option,
    picture_description_options=smolvlm_picture_description
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_option
        ),
        InputFormat.DOCX: WordFormatOption(),   # default options
    }
)
document_path = Path("data/Sample.pdf")


D:\Workspace\RAG_Integration\RAG_System_Unified_Search_Engine\PDF_Extrating\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%%time
result = converter.convert(document_path)
documents = result.document
data_transformed = documents.export_to_markdown(page_no=1)

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 8038.86it/s]
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-08-04 22:42:25,048 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-04 22:42:25,149 [RapidOCR] download_file.py:60: File exists and is valid: D:\Workspace\RAG_Integration\RAG_System_Unified_Search_Engine\PDF_Extrating\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-04 22:42:25,150 [RapidOCR] main.py:63: Using D:\Workspace\RAG_Integration\RAG_System_Unified_Search_Engine\PDF_Extrating\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-

CPU times: total: 1min 50s
Wall time: 1min 25s


Creating Test Data set

In [4]:
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator

llm = LangchainLLMWrapper(
    ChatOllama(
        model="qwen2.5:3b",
        temperature=0,
        format="json"
    )
)

embeddings = LangchainEmbeddingsWrapper(
    OllamaEmbeddings(
        model="nomic-embed-text"
    )
)

generator = TestsetGenerator(
    llm=llm,
    embedding_model=embeddings
)


#response = llm.invoke("Hello")
#print(response.content)

docs = [Document(page_content=data_transformed,metadata={"source": document_path})]


testset = generator.generate_with_langchain_docs(docs,testset_size=20)

C:\Users\Abhideep\AppData\Local\Temp\ipykernel_16236\633466033.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm = LangchainLLMWrapper(
C:\Users\Abhideep\AppData\Local\Temp\ipykernel_16236\633466033.py:17: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = LangchainEmbeddingsWrapper(
Applying OverlapScoreBuilder: 100%|██████████| 1/1 [00:00<00:00, 2202.89it/s]
Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.
Generat

In [8]:
import json

with open("data/ragas_testset.json", "w", encoding="utf-8") as f:
    json.dump(
        testset.to_pandas().to_dict(orient="records"),
        f,
        indent=2,
        ensure_ascii=False
    )